# Exercice 7 - Actifs Pondérés par le Risque de Crédit

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

## Question 1

**LGD** (Loss Given Default) = la perte en cas de défaut, en % de l'exposition. C'est 1 moins le taux de recouvrement. Par exemple si on récupère 40% en cas de défaut, LGD = 60%.

**PD** (Probability of Default) = probabilité que l'emprunteur fasse défaut dans l'année. Estimée par la banque via ses modèles de scoring.

## Question 2

Le coeur de la formule RWA c'est le capital économique K : la VaR 99.9% du taux de défaut (modele Vasicek) moins la perte attendue PD, le tout multiplié par LGD.

En gros : le capital doit couvrir les pertes inattendues (au dela de la perte moyenne qui est couverte par les provisions). Le quantile 99.9% est le scénario de stress. C'est la formule de Vasicek appliquée directement.

In [ ]:
def K_irb(PD, LGD, rho):
    """Capital unitaire IRB"""
    var99 = norm.cdf((norm.ppf(PD) + np.sqrt(rho)*norm.ppf(0.999)) / np.sqrt(1-rho))
    return LGD * (var99 - PD)

## Question 3 - Corrélation réglementaire

rho est imposé par le régulateur, pas estimé par la banque. Il dépend du type d'exposition.

In [ ]:
def rho_corporate(PD):
    return 0.12 * (1-np.exp(-50*PD))/(1-np.exp(-50)) + 0.24 * (1 - (1-np.exp(-50*PD))/(1-np.exp(-50)))

def rho_retail(PD):
    return 0.03 * (1-np.exp(-35*PD))/(1-np.exp(-35)) + 0.16 * (1 - (1-np.exp(-35*PD))/(1-np.exp(-35)))

In [ ]:
PD_range = np.linspace(0.001, 0.15, 200)

plt.figure(figsize=(8, 4))
plt.plot(PD_range*100, [rho_corporate(p) for p in PD_range], 'b-', lw=2, label='Corporate')
plt.plot(PD_range*100, [rho_retail(p) for p in PD_range], 'r-', lw=2, label='Retail')
plt.axhline(0.15, color='g', ls='--', alpha=0.5, label='Hypothécaire')
plt.axhline(0.04, color='purple', ls='--', alpha=0.5, label='Renouvelable')
plt.xlabel('PD (%)')
plt.ylabel('rho')
plt.title('Corrélation réglementaire')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 3a - f(x,y) = (1 - exp(-xy)) / (1 - exp(-y))

C'est la fonction qui sert de poids pour interpoler entre les bornes de rho.

In [ ]:
def f_xy(x, y):
    return (1 - np.exp(-x*y)) / (1 - np.exp(-y))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

x = np.linspace(0.001, 1, 100)
for yy in [10, 35, 50]:
    ax1.plot(x, f_xy(x, yy), label=f'y={yy}')
ax1.set_xlabel('x'); ax1.set_ylabel('f(x,y)')
ax1.set_title('f vs x'); ax1.legend(); ax1.grid(alpha=0.3)

y = np.linspace(1, 80, 100)
for xx in [0.01, 0.05, 0.2]:
    ax2.plot(y, f_xy(xx, y), label=f'x={xx}')
ax2.set_xlabel('y'); ax2.set_ylabel('f(x,y)')
ax2.set_title('f vs y'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 3b - Bornes

f(x,y) varie entre 0 et 1. Donc pour les corporates, rho est borné entre 0.12 (quand f=1, PD élevée) et 0.24 (quand f=0, PD faible). Pour le retail c'est entre 0.03 et 0.16.

La corrélation baisse quand PD augmente : les mauvais emprunteurs font defaut pour des raisons plus idiosyncratiques.

In [ ]:
print(f"Corporate : rho ∈ [{rho_corporate(0.20):.2f}, {rho_corporate(0.0001):.2f}]")
print(f"Retail    : rho ∈ [{rho_retail(0.20):.2f}, {rho_retail(0.0001):.2f}]")

## Question 4 - Maturity Adjustment

In [ ]:
def MA(M, PD):
    b = (0.11852 - 0.05478*np.log(PD))**2
    return (1 + (M - 2.5)*b) / (1 - 1.5*b)

In [ ]:
M_range = np.linspace(0.5, 5, 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# 4a
ax1.plot(M_range, [MA(m, 0.002) for m in M_range], 'b-', lw=2, label='PD=0.2%')
ax1.plot(M_range, [MA(m, 0.04) for m in M_range], 'r-', lw=2, label='PD=4%')
ax1.axhline(1, color='grey', ls='--', alpha=0.5)
ax1.set_xlabel('M (années)'); ax1.set_ylabel('MA')
ax1.set_title('MA vs M'); ax1.legend(); ax1.grid(alpha=0.3)

# 4b
PD_plot = np.linspace(0.001, 0.05, 100)
ax2.plot(PD_plot*100, [MA(3, pd) for pd in PD_plot], 'b-', lw=2)
ax2.axhline(1, color='grey', ls='--', alpha=0.5)
ax2.set_xlabel('PD (%)'); ax2.set_ylabel('MA')
ax2.set_title('MA vs PD (M=3)'); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 4c

Le MA corrige le fait que Vasicek c'est un modele à 1 an. Pour les prets plus longs, il y a un risque de migration (downgrade avant defaut) qui augmente le risque. MA = 1 quand M = 1 an (pas de correction nécessaire), MA > 1 pour M > 1 (pénalité), MA < 1 pour M < 1.

Le parametre b est plus grand pour les faibles PD : les bons emprunteurs sont plus sensibles à la maturité car ils ont plus de "marge de downgrade" avant le defaut.

In [ ]:
# verif MA(M=1) = 1
print(f"MA(M=1, PD=1%) = {MA(1, 0.01):.4f}")
print(f"MA(M=1, PD=5%) = {MA(1, 0.05):.4f}")

## Question 5 - SF = 1.06

Le scaling factor de 1.06 c'est un coussin de sécurité ajouté par le comité de Bâle. Le but c'est que le passage à l'IRB ne réduise pas le capital global du systeme bancaire par rapport à l'ancien regime (Bâle I). C'est une marge de 6% pour compenser les incertitudes de modèle.

## Question 6 - MCR = 12.5

12.5 = 1 / 8%. Ca vient du ratio de solvabilité : FP / RWA >= 8%.

En multipliant le capital K par 12.5, on convertit en RWA tel que 8% * RWA = K * EAD. C'est juste une convention pour que la contrainte reglementaire soit écrite en terme de ratio à 8%.

In [ ]:
# exemple complet
PD_ex = 0.02; LGD_ex = 0.45; EAD = 1_000_000; M_ex = 3
SF = 1.06; MCR = 12.5

rho_ex = rho_corporate(PD_ex)
K_ex = K_irb(PD_ex, LGD_ex, rho_ex)
MA_ex = MA(M_ex, PD_ex)

RWA = K_ex * MA_ex * SF * MCR * EAD

print(f"rho = {rho_ex:.4f}, K = {K_ex:.4f}, MA = {MA_ex:.4f}")
print(f"RWA = {RWA:,.0f} €")
print(f"Capital requis (8% x RWA) = {0.08*RWA:,.0f} € = {0.08*RWA/EAD*100:.1f}% de l'EAD")